In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

## Positive pairs

In [ ]:
df=pd.read_json('/content/drive/MyDrive/QA data /Langchain/positive_pairs.jsonl', lines=True)
df.head()

,document_id,anchor,positive,framework,source
0,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Example usage of __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",langchain,synthetic_positive_v2
1,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How does __getattr__ work in Python?,"def __getattr__(name: str) -> Any:\n """"""Loo...",langchain,synthetic_positive_v2
2,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How to implement __getattr__?,"def __getattr__(name: str) -> Any:\n """"""Loo...",langchain,synthetic_positive_v2
3,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Best practices for __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",langchain,synthetic_positive_v2
4,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Explain the __getattr__ logic,"def __getattr__(name: str) -> Any:\n """"""Loo...",langchain,synthetic_positive_v2


In [ ]:
## drop column
df.drop(['framework','source'], axis=1, inplace=True)

In [ ]:
df.head()

,document_id,anchor,positive
0,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Example usage of __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo..."
1,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How does __getattr__ work in Python?,"def __getattr__(name: str) -> Any:\n """"""Loo..."
2,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How to implement __getattr__?,"def __getattr__(name: str) -> Any:\n """"""Loo..."
3,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Best practices for __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo..."
4,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Explain the __getattr__ logic,"def __getattr__(name: str) -> Any:\n """"""Loo..."


In [ ]:
## adding a column
df['id'] = range(len(df))
df.head(5)

,document_id,anchor,positive,id
0,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Example usage of __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",0
1,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How does __getattr__ work in Python?,"def __getattr__(name: str) -> Any:\n """"""Loo...",1
2,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,How to implement __getattr__?,"def __getattr__(name: str) -> Any:\n """"""Loo...",2
3,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Best practices for __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",3
4,doc_747162eb6fb4374bc6687eb1d59551a114b264ad,Explain the __getattr__ logic,"def __getattr__(name: str) -> Any:\n """"""Loo...",4


In [ ]:
unique_document_ids = df['document_id'].unique()
id_to_numeric = {doc_id: i for i, doc_id in enumerate(unique_document_ids)}
df['document_id'] = df['document_id'].map(id_to_numeric)
df.head()

,document_id,anchor,positive,id
0,0,Example usage of __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",0
1,0,How does __getattr__ work in Python?,"def __getattr__(name: str) -> Any:\n """"""Loo...",1
2,0,How to implement __getattr__?,"def __getattr__(name: str) -> Any:\n """"""Loo...",2
3,0,Best practices for __getattr__,"def __getattr__(name: str) -> Any:\n """"""Loo...",3
4,0,Explain the __getattr__ logic,"def __getattr__(name: str) -> Any:\n """"""Loo...",4


## Dataset (Corpus)

In [ ]:
## shuffle the df
df = df.sample(frac=1).reset_index(drop=True)
df.head(5)

,document_id,anchor,positive,id
0,56,Example usage of test_all_imports,def test_all_imports() -> None:\n assert se...,280
1,24,How to implement _load_vector_db_qa_with_sourc...,def _load_vector_db_qa_with_sources_chain(\n ...,120
2,107,Example usage of _persist_run,"def _persist_run(self, run: Run) -> None:\n ...",535
3,112,How does __getattr__ work in Python?,"def __getattr__(name: str) -> Any:\n """"""Loo...",564
4,45,How does ChatGroq work in Python?,"class ChatGroq(BaseChatModel):\n r""""""Groq C...",227


In [ ]:
# Split Df Into a 90/10 Train/Test split using scikit-learn train test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1)

In [ ]:
## save train and test to json
train.to_json('train.json', orient='records', lines=True)
test.to_json('test.json', orient='records', lines=True)

In [ ]:
# Convert datasets into dictionary format required by the InformationRetrievalEvaluator

# corpus: maps corpus IDs to their text chunks (documents)
# Format: {corpus_id: text_chunk}
corpus = df.set_index('id')['positive'].to_dict()

# queries: maps query IDs to their questions
# Format: {query_id: question_text}
queries = test.set_index('id')['anchor'].to_dict()

# Create a mapping between queries and their relevant documents
# This tells the evaluator which documents are correct matches for each query
relevant_docs = {}
for q_id, document_id in zip(test["id"], test["document_id"]):
    # Initialize empty list for each query if not already present
    if q_id not in relevant_docs:
        relevant_docs[q_id] = []

    # Find all corpus entries that share the same global_chunk_id
    # This handles cases where multiple questions can refer to the same text chunk
    matching_corpus_ids = [
        cid for cid, chunk in zip(df["id"], df["document_id"])
        if chunk == document_id
    ]
    # Add the matching corpus IDs to the relevant documents for this query
    relevant_docs[q_id].extend(matching_corpus_ids)

In [ ]:
relevant_docs

{754: [752, 751, 750, 754, 753],
 531: [533, 532, 534, 531, 530],
 838: [838, 836, 839, 837, 835],
 539: [535, 536, 539, 538, 537],
 550: [552, 553, 554, 550, 551],
 948: [945, 949, 947, 948, 946],
 200: [200, 204, 201, 202, 203],
 429: [425, 427, 426, 429, 428],
 81: [80, 83, 81, 84, 82],
 364: [361, 364, 360, 363, 362],
 223: [223, 222, 220, 224, 221],
 939: [935, 936, 938, 937, 939],
 47: [47, 48, 46, 45, 49],
 291: [291, 293, 292, 290, 294],
 679: [676, 675, 677, 679, 678],
 185: [188, 187, 189, 185, 186],
 330: [330, 332, 331, 334, 333],
 907: [905, 909, 907, 906, 908],
 836: [838, 836, 839, 837, 835],
 917: [918, 915, 917, 919, 916],
 946: [945, 949, 947, 948, 946],
 415: [419, 416, 417, 418, 415],
 634: [632, 631, 634, 633, 630],
 325: [326, 325, 329, 328, 327],
 804: [800, 804, 803, 802, 801],
 782: [784, 781, 782, 783, 780],
 622: [620, 622, 621, 624, 623],
 892: [891, 894, 892, 890, 893],
 16: [19, 18, 17, 15, 16],
 975: [977, 975, 978, 976, 979],
 436: [436, 435, 437, 439, 4

# Baseline Modelling

In [ ]:
import torch

from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
# Hugging Face model ID
model_id = "microsoft/codebert-base"

# Loading via SentenceTransformer
model = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

## Matryoshka Representation Learning

In [ ]:
# Dimensions of interest
matryoshka_dimensions = [768, 512, 256, 128, 64] # Important: large to small

# Create empty list to hold evaluators
matryoshka_evaluators = []

# Create an evaluator for each above dimension
for dim in matryoshka_dimensions:
    # Define the evaluator
    ir_evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"dim_{dim}",
        truncate_dim=dim,  # Truncate the embeddings to the respective dimension
        score_functions={"cosine": cos_sim},
    )
    # Add to list
    matryoshka_evaluators.append(ir_evaluator)

# Create a sequential evaluator
# Able to run all our dimension specific InformationRetrievalEvaluators sequentially.
evaluator = SequentialEvaluator(matryoshka_evaluators)


## Initial Base Result

In [ ]:
# Evaluate the model
base_results = evaluator(model)

# Print header
print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(base_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {base_results['sequential_score']:1f}")


Base Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.0200       0.0100       0.0100       0.0054       0.0100 
mrr@10                 0.0200       0.0100       0.0100       0.0017       0.0100 
accuracy@1             0.0200       0.0100       0.0100       0.0000       0.0100 
precision@3            0.0200       0.0100       0.0100       0.0000       0.0100 
precision@5            0.0200       0.0100       0.0100       0.0000       0.0100 
recall@3               0.0120       0.0060       0.0060       0.0000       0.0060 
-------------------------------------------------------------------------------------
seq_score: 0.010000


# Training or Finetuning codeBERT

In [ ]:
from sentence_transformers.datasets import ParallelSentencesDataset
from sentence_transformers.losses import CosineSimilarityLoss

In [ ]:
# load model with SDPA for using Flash Attention 2
model = SentenceTransformer(
    model_id,
    model_kwargs={"attn_implementation": "sdpa"},
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="apache-2.0",
        model_name="codeBert Base",
    ),
)

In [ ]:
# Initial Loss
base_loss = MultipleNegativesRankingLoss(model)

# Matryoshka Loss Wrapper
train_loss = MatryoshkaLoss(
    model, base_loss, matryoshka_dims=matryoshka_dimensions
)

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir="codebert-base-shubha",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dim_128_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
)


In [ ]:
from datasets import Dataset

# Explicitly keep ONLY text columns
train_dataset = Dataset.from_pandas(
    train[["anchor", "positive"]].astype(str),
    preserve_index=False
)


In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,  # training dataset
    loss=train_loss,
    evaluator=evaluator,
)

In [ ]:
# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Dim 768 Cosine Accuracy@1,Dim 768 Cosine Accuracy@3,Dim 768 Cosine Accuracy@5,Dim 768 Cosine Accuracy@10,Dim 768 Cosine Precision@1,Dim 768 Cosine Precision@3,Dim 768 Cosine Precision@5,Dim 768 Cosine Precision@10,Dim 768 Cosine Recall@1,Dim 768 Cosine Recall@3,Dim 768 Cosine Recall@5,Dim 768 Cosine Recall@10,Dim 768 Cosine Ndcg@10,Dim 768 Cosine Mrr@10,Dim 768 Cosine Map@100,Dim 512 Cosine Accuracy@1,Dim 512 Cosine Accuracy@3,Dim 512 Cosine Accuracy@5,Dim 512 Cosine Accuracy@10,Dim 512 Cosine Precision@1,Dim 512 Cosine Precision@3,Dim 512 Cosine Precision@5,Dim 512 Cosine Precision@10,Dim 512 Cosine Recall@1,Dim 512 Cosine Recall@3,Dim 512 Cosine Recall@5,Dim 512 Cosine Recall@10,Dim 512 Cosine Ndcg@10,Dim 512 Cosine Mrr@10,Dim 512 Cosine Map@100,Dim 256 Cosine Accuracy@1,Dim 256 Cosine Accuracy@3,Dim 256 Cosine Accuracy@5,Dim 256 Cosine Accuracy@10,Dim 256 Cosine Precision@1,Dim 256 Cosine Precision@3,Dim 256 Cosine Precision@5,Dim 256 Cosine Precision@10,Dim 256 Cosine Recall@1,Dim 256 Cosine Recall@3,Dim 256 Cosine Recall@5,Dim 256 Cosine Recall@10,Dim 256 Cosine Ndcg@10,Dim 256 Cosine Mrr@10,Dim 256 Cosine Map@100,Dim 128 Cosine Accuracy@1,Dim 128 Cosine Accuracy@3,Dim 128 Cosine Accuracy@5,Dim 128 Cosine Accuracy@10,Dim 128 Cosine Precision@1,Dim 128 Cosine Precision@3,Dim 128 Cosine Precision@5,Dim 128 Cosine Precision@10,Dim 128 Cosine Recall@1,Dim 128 Cosine Recall@3,Dim 128 Cosine Recall@5,Dim 128 Cosine Recall@10,Dim 128 Cosine Ndcg@10,Dim 128 Cosine Mrr@10,Dim 128 Cosine Map@100,Dim 64 Cosine Accuracy@1,Dim 64 Cosine Accuracy@3,Dim 64 Cosine Accuracy@5,Dim 64 Cosine Accuracy@10,Dim 64 Cosine Precision@1,Dim 64 Cosine Precision@3,Dim 64 Cosine Precision@5,Dim 64 Cosine Precision@10,Dim 64 Cosine Recall@1,Dim 64 Cosine Recall@3,Dim 64 Cosine Recall@5,Dim 64 Cosine Recall@10,Dim 64 Cosine Ndcg@10,Dim 64 Cosine Mrr@10,Dim 64 Cosine Map@100,Sequential Score
1,1.901100,No log,0.580000,0.610000,0.620000,0.760000,0.580000,0.586667,0.586000,0.355000,0.116000,0.352000,0.586000,0.710000,0.653037,0.618194,0.655251,0.570000,0.600000,0.600000,0.740000,0.570000,0.570000,0.568000,0.349000,0.114000,0.342000,0.568000,0.698000,0.639257,0.606429,0.641833,0.560000,0.570000,0.570000,0.740000,0.560000,0.550000,0.544000,0.345000,0.112000,0.330000,0.544000,0.690000,0.626877,0.592302,0.631882,0.590000,0.610000,0.610000,0.760000,0.590000,0.580000,0.578000,0.365000,0.118000,0.348000,0.578000,0.730000,0.663096,0.624095,0.652654,0.590000,0.600000,0.610000,0.760000,0.590000,0.590000,0.588000,0.366000,0.118000,0.354000,0.588000,0.732000,0.665765,0.621583,0.653924,0.665765
2,0.188700,No log,0.810000,0.840000,0.840000,0.900000,0.810000,0.813333,0.812000,0.439000,0.162000,0.488000,0.812000,0.878000,0.847996,0.831667,0.845202,0.850000,0.870000,0.870000,0.920000,0.850000,0.836667,0.834000,0.443000,0.170000,0.502000,0.834000,0.886000,0.864273,0.865873,0.860350,0.820000,0.870000,0.870000,0.930000,0.820000,0.826667,0.822000,0.449000,0.164000,0.496000,0.822000,0.898000,0.864064,0.854583,0.853106,0.800000,0.840000,0.850000,0.920000,0.800000,0.813333,0.812000,0.446000,0.160000,0.488000,0.812000,0.892000,0.853159,0.832944,0.841822,0.730000,0.750000,0.760000,0.880000,0.730000,0.730000,0.736000,0.426000,0.146000,0.438000,0.736000,0.852000,0.797364,0.762500,0.785014,0.797364
3,0.095900,No log,0.840000,0.860000,0.860000,0.930000,0.840000,0.836667,0.834000,0.448000,0.168000,0.502000,0.834000,0.896000,0.868828,0.859524,0.863604,0.850000,0.860000,0.870000,0.950000,0.850000,0.840000,0.842000,0.453000,0.170000,0.504000,0.842000,0.906000,0.877431,0.869940,0.869074,0.850000,0.880000,0.890000,0.930000,0.850000,0.840000,0.840000,0.451000,0.170000,0.504000,0.840000,0.902000,0.875364,0.872083,0.866457,0.830000,0.880000,0.880000,0.920000,0.830000,0.840000,0.842000,0.451000,0.166000,0.504000,0.842000,0.902000,0.872493,0.859762,0.864234,0.780000,0.810000,0.810000,0.940000,0.780000,0.786667,0.786000,0.449000,0.156000,0.472000,0.786000,0.8980

TrainOutput(global_step=60, training_loss=0.48467007279396057, metrics={'train_runtime': 988.7379, 'train_samples_per_second': 3.641, 'train_steps_per_second': 0.061, 'total_flos': 0.0, 'train_loss': 0.48467007279396057, 'epoch': 4.0})

In [ ]:
# Save the best model based on our eval_dim_128_cosine_ndcg@10 criteria
trainer.save_model()

#### Uploading to huggingface

In [ ]:
# Upload model to hub
trainer.model.push_to_hub("codebert-embed-base-dense-retriever")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...jqpauz2/model.safetensors:   0%|          |  550kB /  499MB            

'https://huggingface.co/killdollar/codebert-embed-base-dense-retriever/commit/9594580ae943039d0b85feb304404f9b2bb203ce'

## Finetuned codeBERT model evaluation

In [ ]:
fine_tuned_model = SentenceTransformer(
    args.output_dir, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Evaluate the model
ft_results = evaluator(fine_tuned_model)

# Print header
print("Fine Tuned Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]
# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(ft_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {ft_results['sequential_score']:1f}")

Fine Tuned Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.8678       0.8766       0.8738       0.8754       0.8407 
mrr@10                 0.8518       0.8646       0.8670       0.8629       0.8096 
accuracy@1             0.8300       0.8400       0.8400       0.8400       0.7800 
precision@3            0.8300       0.8433       0.8367       0.8333       0.7833 
precision@5            0.8280       0.8420       0.8420       0.8380       0.7840 
recall@3               0.4980       0.5060       0.5020       0.5000       0.4700 
-------------------------------------------------------------------------------------
seq_score: 0.840715
